In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)


class NeuralNetArchitecture(nn.Module):
    def __init__(self, in_dim: int, out_dim: int) -> None:
        super().__init__()
        self.layer_1 = nn.Linear(in_dim, 12)
        self.layer_2 = nn.Linear(12, 10)
        self.layer_3 = nn.Linear(10, out_dim)

    def forward(self, x):
        x = F.relu(self.layer_1(x))
        x = F.relu(self.layer_2(x))
        x = self.layer_3(x)

        return x

In [13]:
from skorch import NeuralNetRegressor


class GibbsDuhemInformed(NeuralNetRegressor):
    def __init__(self, *args, lambda_gd: int = 1, **kwargs):
        super().__init__(*args, **kwargs)
        self.lambda_gd = lambda_gd

    def get_loss(self, y_pred, y_true, X=None, training=False):
        # Default loss
        loss = self.criterion_(y_pred, y_true)

        # Gibbs Duhem loss
        gamma1 = y_pred[:, 0]
        dgamma1_dx1 = torch.autograd.grad(
            gamma1,
            X,
            grad_outputs=torch.ones_like(gamma1),
            create_graph=True,
            allow_unused=True,
        )[0][:, 2]

        gamma2 = y_pred[:, 1]
        dgamma2_dx2 = torch.autograd.grad(
            gamma2,
            X,
            grad_outputs=torch.ones_like(gamma2),
            create_graph=True,
            allow_unused=True,
        )[0][:, 5]

        gibbs_duhem_loss = torch.mean((dgamma1_dx1 + dgamma2_dx2) ** 2)

        # Total loss
        loss += self.lambda_gd * gibbs_duhem_loss

        return loss

In [14]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset
data = pd.read_csv("C:/Projetos/biofuels-sciml/data/processed/toy_problem_input_dataset.csv").values

features = data[:, :-2]
labels = data[:, -2:]

X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.1, random_state=42
)

# Scale features
scaler = MinMaxScaler()
scaler.fit(X_train)

# Scaled data
X_train_scaled = torch.tensor(
    scaler.transform(X_train), dtype=torch.float32, requires_grad=True
).to(device)
X_test_scaled = torch.tensor(
    scaler.transform(X_test), dtype=torch.float32, requires_grad=True
).to(device)
y_train = torch.tensor(y_train, dtype=torch.float32).to(device)
y_test = torch.tensor(y_test, dtype=torch.float32).to(device)

In [15]:
net_gd = GibbsDuhemInformed(
    module=NeuralNetArchitecture(X_train_scaled.shape[1], 2),
    lambda_gd=1e-1,
    lr=0.001,
    criterion=nn.MSELoss(),
    optimizer=torch.optim.Adam,
    batch_size=X_train_scaled.shape[0],
    max_epochs=5000,
    device="cuda",
    verbose=1,
    train_split=None, # train_split was set to None because torch does not compute grad during validation
)

net_gd.fit(X_train_scaled, y_train)

# Evaluate a stop early which depends only from train_loss

  epoch    train_loss     dur
-------  ------------  ------
      1        2.8786  0.1122
      2        2.8685  0.0700
      3        2.8585  0.0803
      4        2.8486  0.0708
      5        2.8387  0.0341
      6        2.8289  0.0648
      7        2.8192  0.0429
      8        2.8096  0.0377
      9        2.8000  0.0149
     10        2.7905  0.0146
     11        2.7810  0.0155
     12        2.7715  0.0161
     13        2.7620  0.0165
     14        2.7525  0.0156
     15        2.7430  0.0142
     16        2.7335  0.0146
     17        2.7239  0.0161
     18        2.7143  0.0157
     19        2.7047  0.0142
     20        2.6952  0.0146
     21        2.6859  0.0155
     22        2.6767  0.0144
     23        2.6674  0.0145
     24        2.6582  0.0156
     25        2.6489  0.0152
     26        2.6396  0.0152
     27        2.6302  0.0167
     28        2.6206  0.0144
     29        2.6110  0.0160
     30        2.6013  0.0200
     31        2.5915  0.0150
     32   

<class '__main__.GibbsDuhemInformed'>[initialized](
  module_=NeuralNetArchitecture(
    (layer_1): Linear(in_features=7, out_features=12, bias=True)
    (layer_2): Linear(in_features=12, out_features=10, bias=True)
    (layer_3): Linear(in_features=10, out_features=2, bias=True)
  ),
)

In [25]:
# Evaluation
from sklearn.metrics import root_mean_squared_error

y_pred = net_gd.predict(X_test_scaled)
loss = root_mean_squared_error(y_pred, y_test.cpu().numpy())

print(loss)

1.1156902
